# YOLOv8n Fine-Tuning

Fine-tunes YOLOv8n on the 7-class BC wildlife dataset.
Training uses COCO pretrained weights with online augmentation.

**Classes:** bear, cougar, coyote, deer, elk, moose, raccoon
**Split:** 70/20/10 (2001 train / 572 val / 287 test), seed 42
**Target:** YOLOv8n (nano) to match Hailo-8L deployment constraints.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!cp "/content/drive/MyDrive/bc-dashcam/yolo_dataset.zip" /content/
!cd /content && unzip -q yolo_dataset.zip && echo "UNZIP DONE"
!ls /content/yolo_dataset

## Verify dataset parity

Every image must have a corresponding label file.

In [ ]:
import re
import os

data_yaml = "/content/yolo_dataset/data.yaml"
s = open(data_yaml).read()
s = re.sub(r"^path:.*$", "path: /content/yolo_dataset", s, flags=re.M)
open(data_yaml, "w").write(s)
print(s)

for split in ["train", "valid", "test"]:
    img = len(os.listdir(f"/content/yolo_dataset/{split}/images"))
    lbl = len(os.listdir(f"/content/yolo_dataset/{split}/labels"))
    print(f"{split}: {img} imgs / {lbl} lbls")

In [ ]:
!pip install -q ultralytics wandb
import wandb
wandb.login()

## Train

Augmentation is applied online during training, not baked into files.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

model.train(
    data="/content/yolo_dataset/data.yaml",
    epochs=100,
    imgsz=640,
    batch=32,
    patience=20,
    project="bc-dashcam-yolov8",
    name="run1-7class",
    seed=42,
    device=0,
    pretrained=True,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    fliplr=0.5,
    mosaic=1.0,
    degrees=5.0,
    translate=0.1,
    scale=0.5,
)

## Save weights to Drive

In [ ]:
!cp -r /content/runs/detect/bc-dashcam-yolov8/run1-7class "/content/drive/MyDrive/bc-dashcam/run1-7class"
print("Saved to Drive.")

## Review training results

In [ ]:
from IPython.display import Image, display

base = "/content/runs/detect/bc-dashcam-yolov8/run1-7class"
for f in ["confusion_matrix.png", "results.png", "val_batch0_pred.jpg"]:
    print(f)
    display(Image(f"{base}/{f}"))